# Installation Check

This notebook is a lightweight smoke test for the `collective_posterior` environment.

It checks:
- core package imports and versions
- local priors and simulators for `WF`, `GLU`, `SLCP`, and `EVO_SIM`
- loading a shipped pretrained posterior pickle
- a small `CollectivePosterior` run

Run it from the repository root `Posterior/collective_posterior`.

This notebook does not test `phylo_abc.ipynb`, which still depends on the external `GenomeRearrangement` package.

In [1]:
from pathlib import Path
import platform
import pickle
import sys

import matplotlib
import networkx as nx
import numpy as np
import pandas as pd
import sbi
import scipy
import seaborn as sns
import torch
import sbibm

from collective_posterior import CollectivePosterior
from evo_sim import evo_sim
from inference_utils import get_prior
from simulators import GLU, SLCP, WF, WF_wrapper

repo_root = Path.cwd()
assert (repo_root / "requirements.txt").exists(), "Run this notebook from Posterior/collective_posterior"

print(f"Python: {platform.python_version()}")
print(f"Torch: {torch.__version__}")
print(f"sbi: {sbi.__version__}")
print(f"NumPy: {np.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"NetworkX: {nx.__version__}")

Python: 3.10.20
Torch: 2.5.1+cpu
sbi: 0.25.0
NumPy: 1.24.4
SciPy: 1.11.4
Pandas: 2.1.4
Matplotlib: 3.8.2
Seaborn: 0.13.2
NetworkX: 3.2.1


In [2]:
theta_wf = torch.tensor([-1.5, -5.0, -6.0], dtype=torch.float32)
theta_evo = torch.tensor([-2.0, -2.0, -2.0, -6.0, -6.0, -6.0], dtype=torch.float32)
theta_glu = get_prior("GLU").sample()
theta_slcp = get_prior("SLCP").sample()

x_wf = WF(theta_wf)
x_evo = evo_sim(theta_evo)
x_glu = GLU(theta_glu)
x_slcp = SLCP(theta_slcp)

assert x_wf.shape == (12,), f"Unexpected WF shape: {x_wf.shape}"
assert x_evo.ndim == 1, f"Unexpected EVO_SIM ndim: {x_evo.ndim}"
assert x_glu.shape == (1, 10), f"Unexpected GLU shape: {x_glu.shape}"
assert x_slcp.shape == (1, 8), f"Unexpected SLCP shape: {x_slcp.shape}"

print("WF shape:", tuple(x_wf.shape))
print("EVO_SIM shape:", tuple(x_evo.shape))
print("GLU shape:", tuple(x_glu.shape))
print("SLCP shape:", tuple(x_slcp.shape))
print("WF prior sample shape:", tuple(get_prior("WF").sample().shape))

WF shape: (12,)
EVO_SIM shape: (30,)
GLU shape: (1, 10)
SLCP shape: (1, 8)
WF prior sample shape: (3,)


/home/jupyter-nadavbennun/.conda/envs/collective/lib/python3.10/site-packages/sbibm/tasks/slcp/task.py:86: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3683.)
  ).T


In [3]:
posterior_path = repo_root / "WF" / "posteriors" / "posterior_WF_10000_20.pkl"
assert posterior_path.exists(), f"Missing posterior file: {posterior_path}"

with open(posterior_path, "rb") as handle:
    posterior = pickle.load(handle)

posterior_samples = posterior.set_default_x(x_wf).sample((4,))
assert posterior_samples.shape == (4, 3), f"Unexpected posterior sample shape: {posterior_samples.shape}"

print("Loaded posterior:", posterior_path.name)
print("Posterior sample shape:", tuple(posterior_samples.shape))

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


  0%|          | 0/4 [00:00<?, ?it/s]

Loaded posterior: posterior_WF_10000_20.pkl
Posterior sample shape: (4, 3)


In [4]:
Xs = WF_wrapper(reps=3, parameters=theta_wf)

cp = CollectivePosterior(
    prior=get_prior("WF"),
    Xs=Xs,
    amortized_posterior=posterior,
    log_C=1.0,
    epsilon=-10,
    n_eval=128,
)

theta_test = cp.prior.sample((8,))
logp = cp.log_prob(theta_test)
assert logp.shape == (8,), f"Unexpected log_prob shape: {logp.shape}"
assert torch.isfinite(logp).all(), "Collective posterior log_prob returned non-finite values"

log_C = cp.get_log_C(n_reps=1, S=0.05)
assert torch.isfinite(log_C), f"Non-finite log_C: {log_C}"

cp_samples = cp.sample_via_sir_jitter(
    n_draws=128,
    n_final=16,
    bandwidth_scale=0.1,
    excess_quantile=0.25,
)
assert cp_samples.shape == (16, 3), f"Unexpected collective sample shape: {cp_samples.shape}"

print("Collective log_prob shape:", tuple(logp.shape))
print("Estimated log_C:", float(log_C))
print("Collective sample shape:", tuple(cp_samples.shape))
print("Installation check passed.")

Evaluating 128 samples: 100%|██████████| 3/3 [00:00<00:00, 289.77it/s]

Sampled 16 points with jitter and reflection. ESS = 4.388702392578125
Collective log_prob shape: (8,)
Estimated log_C: 4.51664400100708
Collective sample shape: (16, 3)
Installation check passed.
